# Fine-Tuning LLM untuk Sentiment Analysis

**Deskripsi**: Tutorial ini mendemonstrasikan cara melakukan *fine-tuning* (penyetelan halus) model bahasa pra-terlatih (*pretrained language model*) menggunakan **Hugging Face Transformers** untuk tugas klasifikasi sentimen teks Bahasa Indonesia.

**Tujuan Pembelajaran**:
- Memahami konsep *transfer learning* dan *fine-tuning*
- Mampu memuat dataset sentimen dari Hugging Face Datasets
- Mampu melakukan fine-tuning tokenizer dan model transformer
- Mampu mengevaluasi model menggunakan metrik seperti akurasi, precision, recall, dan F1-score
- Mampu menyimpan dan menggunakan model hasil fine-tuning

---## 1. Instalasi Dependensi

In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn seaborn matplotlib pandas tqdm

---## 2. Import Library

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from datasets import load_dataset

import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

---## 3. Load Dataset Sentimen Bahasa Indonesia

In [ ]:
# Load dataset
dataset = load_dataset("sepidmnorozy/Indonesian_sentiment")

print(f"Dataset splits: {dataset.keys()}")
print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print(f"Test size: {len(dataset['test'])}")

In [ ]:
# Lihat contoh data
label_map = {0: "NEGATIVE", 1: "POSITIVE"}

print("=== Contoh Data Training ===")
for i in range(5):
    sample = dataset['train'][i]
    print(f"[{i}] Label: {sample['label']} ({label_map[sample['label']]})")
    print(f"    Teks: {sample['text'][:100]}...")
    print()

In [ ]:
# Periksa distribusi label
train_labels = dataset['train'].to_pandas()['label']
val_labels = dataset['validation'].to_pandas()['label']
test_labels = dataset['test'].to_pandas()['label']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for i, (labels, title) in enumerate([
    (train_labels, 'Train'),
    (val_labels, 'Validation'),
    (test_labels, 'Test')
]):
    counts = labels.value_counts().sort_index()
    axes[i].bar(['Negative (0)', 'Positive (1)'], counts.values, color=['#e74c3c', '#2ecc71'])
    axes[i].set_title(title)
    axes[i].set_ylabel('Jumlah')
    for j, v in enumerate(counts.values):
        axes[i].text(j, v + 20, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---## 4. Tokenisasi

In [ ]:
MODEL_NAME = "indolem/indobertweet-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, model_max_length=MAX_LENGTH)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max length: {tokenizer.model_max_length}")

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=False,
        max_length=MAX_LENGTH
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Tokenization complete!")
print(f"Features: {tokenized_dataset['train'].column_names}")

---## 5. Load Model

In [ ]:
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")

---## 6. Konfigurasi Training

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    
    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        labels, predictions, average=None
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'precision_neg': precision_per_class[0],
        'recall_neg': recall_per_class[0],
        'f1_neg': f1_per_class[0],
        'precision_pos': precision_per_class[1],
        'recall_pos': recall_per_class[1],
        'f1_pos': f1_per_class[1],
    }

In [ ]:
output_dir = "indo-sentiment-finetuned"

training_args = TrainingArguments(
    output_dir=output_dir,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    report_to="none",
    seed=42,
)

print(f"Output directory: {output_dir}")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Epochs: {training_args.num_train_epochs}")

---## 7. Fine-Tuning

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer initialized!")

In [ ]:
# Jalankan fine-tuning
train_result = trainer.train()
print("Training complete! ✅")
print(f"Training loss: {train_result.training_loss:.4f}")

---## 8. Evaluasi

In [ ]:
test_results = trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")

print("=== Hasil Evaluasi pada Test Set ===")
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

In [ ]:
predictions = trainer.predict(tokenized_dataset["test"])
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print("=== Classification Report ===")
print(classification_report(true_labels, pred_labels, target_names=["NEGATIVE", "POSITIVE"]))

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NEGATIVE', 'POSITIVE'],
            yticklabels=['NEGATIVE', 'POSITIVE'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Sentiment Analysis')
plt.show()

---## 9. Kesimpulan

Dalam tutorial ini, kita telah berhasil:

1. ✅ Memahami konsep fine-tuning
2. ✅ Memuat dan memproses dataset
3. ✅ Melakukan tokenisasi
4. ✅ Fine-tuning IndoBERTweet
5. ✅ Mengevaluasi model
6. ✅ Menyimpan dan memuat model

**Takeaways Penting**:
- Fine-tuning memungkinkan model bahasa umum beradaptasi ke tugas spesifik
- Model BERT seperti IndoBERTweet sangat efektif untuk Bahasa Indonesia
- Evaluasi komprehensif penting untuk memahami performa model